# Image Cluster Notebook
This notebook is designed to create labels for NAC images through K-Means clustering. It does 2 main passes: the initial K-means pass where the algorithm infers pixel grouping from their pixel values. A second pass is then done where the user must select the cluster values that belong to class 0 (no crater) and 1 (crater). These are then submitted to the display function and shown at the end of the notebook. 

## Imports

In [ ]:
# PROJ must be configured before rasterio/localtileserver are imported.
import os
os.environ["PROJ_IGNORE_CELESTIAL_BODY"] = "YES"

from pathlib import Path
import sys
import ipysheet
from IPython.display import Markdown, display
import ipywidgets
import leafmap
import numpy
import pandas
import rasterio
from rasterio.windows import Window
from localtileserver import TileClient, get_leaflet_tile_layer
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from ipyleaflet import WidgetControl

repo_root = Path.cwd().parent
repo_root_str = str(repo_root).replace('/panfs/ccds02/nobackup', '/explore/nobackup')
repo_root = Path(repo_root_str)
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )

sys.path.insert(0, str(repo_root))

from model.clustering.Clusterer import Clusterer
from model.clustering.ImageHelperSingleBand import ImageHelper
from model.clustering.clustering_display_utils import display_images_labels, display_images_binary_labels
from model.clustering.clustering_backend_utils import crop_center, handleClick, relabel, updateDict, updateList

# Configuration

`inFile`: Input single-band NAC file to perform clustering on. 

`noDataValue`: Nodata value to ignore in clustering. This is vital for the clustering algorithm to properly capture the valid data distribution. 

`numClusters`: Number of K-means clusters to create. Higher values result in a more noisy output (making it harder to discern between cluster groups by eye), while lower values create more homogenous clusters (which makes for poor labels). 

`cropSize`: Target crop size of input image; image is cropped to size in the center to improve performance of the algorithm and display tools. 

In [ ]:
# Original full-resolution lunar raster.
inFile = "/explore/nobackup/projects/lfm/Benchmarks/Craters/NAC_PHO_E064S3160/NAC_DTM_NEWCRATER6_M1219245090_80CM.TIF"

# Outputs are written here.
outDirectory = ""

NAC_NODATA = -3.40282265508890445e+38
noDataValue = NAC_NODATA

# Number of K-Means clusters to use
numClusters = 20

# Crop size of input image, clustering will run on this smaller image for demonstration purposes
cropSize = 512

## Path setup

In [ ]:
inFile = Path(inFile)

outDirectory = Path(outDirectory)
outDirectory.mkdir(parents=True, exist_ok=True)

clippedInputFile = outDirectory / (
    f"{inFile.stem}-clip-{cropSize}{inFile.suffix}"
)
labelsFile = outDirectory / (
    f"{inFile.stem}-clip-{cropSize}-labels{inFile.suffix}"
)
clusterMapFile = outDirectory / (
    f"{inFile.stem}-clip-{cropSize}-cluster-map{inFile.suffix}"
)

# Step 1: Clip the input raster

In [ ]:
# This is intentionally performed BEFORE ImageHelper ingestion or clustering.
# The full input TIFF is never passed to Clusterer.getClusters().
crop_center(
    src_path=inFile,
    dst_path=clippedInputFile,
    size=cropSize,
)

print(f"Full input:    {inFile}")
print(f"Clipped input: {clippedInputFile}")

# Step 2: Ingest ONLY the clipped raster

In [ ]:
inHelper = ImageHelper()
inHelper.initFromFile(
    inputFile=clippedInputFile,
    noDataValue=noDataValue,
)

print(f"Clustering input shape: {inHelper.getBand().shape}")

# Step 3: Generate first-pass clusters on the clipped raster

These clusters will have many different groupings, equal to numClusters. The second pass will narrow them down to only 2 classes (no crater, crater). 

In [ ]:
# Add singleton dimension for the single input band: (H, W) -> (1, H, W).
# For cropSize=512, clustering operates on only 512x512 pixels.
labels = Clusterer.getClusters(
    bands=numpy.expand_dims(inHelper.getBand(), axis=0),
    numClusters=numClusters,
)

# Because inHelper was created from clippedInputFile, the label GeoTIFF is
# automatically written with the same clipped extent/transform/CRS.
labelsDs = Clusterer.labelsToGeotiff(
    inHelper._dataset,
    labelsFile,
    labels,
)

lHelper = ImageHelper()
lHelper.initFromDataset(labelsDs, noDataValue)

print(f"Clipped labels: {labelsFile}")

# Step 4: Display clipped image + clipped labels

In [ ]:
m, legend_control = display_images_labels(clippedInputFile, labelsFile, labels, inHelper, lHelper)
display(m)

## Update the labels
Select multiple values by clicking the mouse or using the arrow keys while pressing Shift, Control, or Command.

In [ ]:
opts = list(numpy.unique(labels))

sl = ipywidgets.SelectMultiple(
    options=opts,
    layout=ipywidgets.Layout(height="200px", width="150px"),
)

bt = ipywidgets.ToggleButtons(
    options=["Select:", "Next", "Done", "Start Over"],
    value="Select:",
)

output = ipywidgets.Output()
display(sl, bt, output)
table = {}
bt.observe(lambda change: handleClick(change, output, sl, bt, opts, table), names="value")

## Edit the groups
Edit cluster IDs in each group. When finished, proceed to the next cell.

In [ ]:
strTab = {}

for item in table:
    strTab[item] = ", ".join(str(i) for i in table[item])

df = pandas.DataFrame(strTab.items(), columns=["Class", "Cluster ID"])
sheet = ipysheet.from_dataframe(df)
sheet.column_width = [1, 5]
sheet

In [ ]:
editedDf = ipysheet.to_dataframe(sheet)
strClusters = editedDf.to_dict()["Cluster ID"]

finalClusters = {}

for key in strClusters:
    strCluster = strClusters[key]
    finalClusters[int(key)] = [
        int(i.strip()) for i in strCluster.split(",") if i.strip()
    ]

print(finalClusters)
newClusters = relabel(labels, finalClusters)

## Review the updated map

In [ ]:
m = display_images_binary_labels(m, inHelper, clusterMapFile, labelsFile, newClusters)
display(m)